# 08 — Assembling the Full GPT Model

**Lecture goal:** stack everything from notebooks 04–07 — embeddings, many transformer blocks, a final normalization, and an output layer — into one complete, working `GPTModel`, and understand every number in its configuration.

## The full architecture, top to bottom

```
token IDs  ->  token embedding  +  positional embedding
           ->  dropout
           ->  TransformerBlock x N   (each: attention + feed-forward, each wrapped in LayerNorm + residual)
           ->  final LayerNorm
           ->  output head (Linear)
           ->  logits  (a score for every possible next token, at every position)
```

Every piece except the very first (embeddings) and the very last (output head) is a `TransformerBlock`, repeated `N` times — `N = 12` for the smallest GPT-2. Let's define a configuration describing all these numbers, build the model, and run data through it.

## Two configurations: the real thing, and a fast one to experiment with

Real GPT-2 (the smallest of the four GPT-2 sizes OpenAI released, "124M" referring to its ~124 million parameters) uses:

- `vocab_size = 50257` — matches our BPE tokenizer from notebook 02.
- `context_length = 1024` — max number of tokens of history the model can look at.
- `embedding_dim = 768` — size of each token's vector representation.
- `num_heads = 12`, `num_layers = 12` — 12 attention heads per block, 12 blocks stacked.
- `dropout_rate = 0.1`
- `qkv_bias = False` — whether the Q/K/V linear layers include a bias term (GPT-2's original checkpoint does include biases here, which matters when we load real pretrained weights in notebook 11).

Training a 124-million-parameter model from scratch, on a laptop CPU, on our tiny `the-verdict.txt` corpus, would technically work but take a long time per step. Since notebooks 08–10 are about *understanding the mechanics* (not producing a great model — that needs vastly more data and compute than we have), we'll define a much smaller configuration with the exact same architecture, and use that for our own from-scratch training in notebook 10. We'll come back to the real 124M configuration in notebook 11, where we load OpenAI's already-trained weights instead of training from scratch ourselves.

In [1]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "embedding_dim": 768,
    "num_heads": 12,
    "num_layers": 12,
    "dropout_rate": 0.1,
    "qkv_bias": True,
}

# Same architecture, much smaller — fast enough to train from scratch on a CPU in notebook 10.
GPT_CONFIG_SMALL = {
    "vocab_size": 50257,
    "context_length": 128,
    "embedding_dim": 256,
    "num_heads": 4,
    "num_layers": 4,
    "dropout_rate": 0.1,
    "qkv_bias": False,
}

## Building `GPTModel`

We reuse `TransformerBlock` and `LayerNorm` exactly as built in notebook 07. `GPTModel` itself is mostly bookkeeping: embeddings in, a stack of blocks, normalize, project to vocabulary size out.

In [2]:
import torch
import torch.nn as nn


class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        batch_size, num_tokens, d_in = x.shape

        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        queries = queries.view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys = keys.view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values = values.view(batch_size, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        attention_scores = queries @ keys.transpose(2, 3)
        attention_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )
        attention_weights = torch.softmax(attention_scores / self.head_dim ** 0.5, dim=-1)
        attention_weights = self.dropout(attention_weights)

        context_vectors = (attention_weights @ values).transpose(1, 2)
        context_vectors = context_vectors.contiguous().view(batch_size, num_tokens, self.d_out)
        return self.out_proj(context_vectors)


class LayerNorm(nn.Module):
    def __init__(self, embedding_dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(embedding_dim))
        self.shift = nn.Parameter(torch.zeros(embedding_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        normalized = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * normalized + self.shift


class FeedForward(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(embedding_dim, 4 * embedding_dim),
            nn.GELU(),
            nn.Linear(4 * embedding_dim, embedding_dim),
        )

    def forward(self, x):
        return self.layers(x)


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attention = MultiHeadAttention(
            d_in=cfg["embedding_dim"],
            d_out=cfg["embedding_dim"],
            context_length=cfg["context_length"],
            dropout=cfg["dropout_rate"],
            num_heads=cfg["num_heads"],
            qkv_bias=cfg["qkv_bias"],
        )
        self.feed_forward = FeedForward(cfg["embedding_dim"])
        self.norm1 = LayerNorm(cfg["embedding_dim"])
        self.norm2 = LayerNorm(cfg["embedding_dim"])
        self.dropout = nn.Dropout(cfg["dropout_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.attention(x)
        x = self.dropout(x)
        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.feed_forward(x)
        x = self.dropout(x)
        x = x + shortcut

        return x

In [3]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.token_embedding = nn.Embedding(cfg["vocab_size"], cfg["embedding_dim"])
        self.position_embedding = nn.Embedding(cfg["context_length"], cfg["embedding_dim"])
        self.dropout = nn.Dropout(cfg["dropout_rate"])

        self.transformer_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["num_layers"])]
        )

        self.final_norm = LayerNorm(cfg["embedding_dim"])
        self.out_head = nn.Linear(cfg["embedding_dim"], cfg["vocab_size"], bias=False)

    def forward(self, token_ids):
        batch_size, num_tokens = token_ids.shape

        token_embeds = self.token_embedding(token_ids)
        positions = torch.arange(num_tokens, device=token_ids.device)
        pos_embeds = self.position_embedding(positions)

        x = token_embeds + pos_embeds
        x = self.dropout(x)
        x = self.transformer_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)

        return logits

`self.transformer_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["num_layers"])])` builds `num_layers` *separate, independently initialized and independently learnable* copies of `TransformerBlock`, chained so data flows through block 1, then block 2, ... then block `N` — `nn.Sequential` simply calls each in turn, feeding each one's output into the next one's input.

## Running data through it

Let's build the small model and pass a real batch of token IDs through it, reusing the `DataLoader` machinery from notebook 03.

In [4]:
import tiktoken
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, text, tokenizer, context_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(text)
        for start in range(0, len(token_ids) - context_length, stride):
            self.input_ids.append(torch.tensor(token_ids[start : start + context_length]))
            self.target_ids.append(torch.tensor(token_ids[start + 1 : start + context_length + 1]))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(text, batch_size=4, context_length=256, stride=128, shuffle=True, drop_last=True):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(text, tokenizer, context_length, stride)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last)


with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

dataloader = create_dataloader_v1(
    raw_text, batch_size=2, context_length=GPT_CONFIG_SMALL["context_length"],
    stride=GPT_CONFIG_SMALL["context_length"], shuffle=False,
)
inputs, targets = next(iter(dataloader))
print("Input batch shape:", inputs.shape)

Input batch shape: torch.Size([2, 128])


In [5]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_SMALL)

logits = model(inputs)
print("Logits shape:", logits.shape)

Logits shape: torch.Size([2, 128, 50257])


`(2, 128, 50257)`: 2 examples in the batch, 128 token positions each, and — this is the new part — **50,257 numbers per position**, one raw score ("logit") for *every possible token in the vocabulary*, representing how strongly the model currently favors that token as "what comes next" at that position.

These are **not** probabilities yet (they aren't constrained to be positive or sum to 1) — they're the raw, unnormalized output of the final linear layer. Turning them into an actual next-token prediction (and generated text) is exactly what notebook 09 covers.

## How big is this model?

Let's count the total number of learnable numbers (parameters) in our small model, and compare against what the real 124M-parameter configuration would allocate — cheap to check, since simply *constructing* a model (allocating its weight tensors) is fast; it's actually *training* one that's expensive.

In [6]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


print(f"GPT_CONFIG_SMALL parameter count: {count_parameters(model):,}")

torch.manual_seed(123)
model_124m = GPTModel(GPT_CONFIG_124M)
print(f"GPT_CONFIG_124M parameter count:  {count_parameters(model_124m):,}")

del model_124m  # free the memory - we're not using this one until notebook 11

GPT_CONFIG_SMALL parameter count: 28,920,832


GPT_CONFIG_124M parameter count:  163,037,184


Our small model has about 29 million parameters — most of that isn't the transformer blocks at all, it's the token-embedding table and output head (each roughly `50257 x 256 = 12.9M` numbers, dominated by the vocabulary size, not our chosen `embedding_dim`). Still, that's small enough to train several passes over a tiny text file on a CPU in reasonable time. The full-size configuration has over 160 million (somewhat more than the commonly quoted "124M" because our simple implementation doesn't share/"tie" the token-embedding and output-head weight matrices the way the official GPT-2 checkpoint does — a detail we'll revisit in notebook 11 when loading real pretrained weights).

## Recap

- `GPTModel` = token + positional embeddings → dropout → a stack of `num_layers` `TransformerBlock`s → final `LayerNorm` → linear output head.
- The output, `logits`, has shape `(batch_size, num_tokens, vocab_size)` — one score per possible next token, at every position in every example.
- We defined two configs: `GPT_CONFIG_124M` (the real GPT-2 small architecture) and `GPT_CONFIG_SMALL` (same architecture, scaled down for fast experimentation on a CPU) — we'll train the small one ourselves in notebook 10, and load real pretrained weights into the full-size one in notebook 11.

### What's next

We can now run a forward pass, but the model can't yet *write* anything — it just spits out one big table of scores. In notebook 09, we'll turn those logits into an actual next-token choice, and repeat that process to generate whole passages of text.